In [ ]:
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import FloatSlider, IntSlider, VBox, HTML, interactive_output, Layout, GridBox
from IPython.display import display

# ============================================================
# WIENER PROCESS AND WHITE NOISE
# ============================================================

# ============================================================
# RANDOM NUMBERS
# ============================================================

np.random.seed(117)

M_max = 200
N_max = 2000

Z = np.random.randn(M_max, N_max)

# ============================================================
# SHORT DOCUMENTATION
# ============================================================

documentation = HTML("""
<div style="
    font-family:Arial, sans-serif;
    font-size:16px;
    line-height:1.40;
    width:1050px;
">

<div style="
    font-size:22px;
    font-weight:bold;
    color:#12388c;
    margin-bottom:8px;
">
Wiener Process and White Noise
</div>

<div style="margin-bottom:4px;">
<b>Wiener process:</b> W(t) is obtained by accumulating independent Gaussian increments.
</div>

<div style="margin-bottom:4px;">
The increments satisfy ΔW ~ N(0, Δt), so E{W(t)} = 0 and Var{W(t)} = t.
</div>

<div style="margin-bottom:4px;">
The finite-difference derivative ΔW/Δt becomes increasingly irregular as Δt decreases.
</div>

<div>
<b>This notebook:</b> illustrates the connection between Wiener motion and ideal white noise.
</div>

</div>
""")

# ============================================================
# SLIDERS
# ============================================================

slider_style = {'description_width': '0px'}
slider_layout = Layout(width='160px')

dt_slider = FloatSlider(
    min=0.005,
    max=0.05,
    step=0.005,
    value=0.02,
    description=' ',
    readout=False,
    continuous_update=True,
    style=slider_style,
    layout=slider_layout
)

T_slider = FloatSlider(
    min=2.0,
    max=10.0,
    step=1.0,
    value=6.0,
    description=' ',
    readout=False,
    continuous_update=True,
    style=slider_style,
    layout=slider_layout
)

M_slider = IntSlider(
    min=20,
    max=200,
    step=20,
    value=100,
    description=' ',
    readout=False,
    continuous_update=True,
    style=slider_style,
    layout=slider_layout
)

show_slider = IntSlider(
    min=1,
    max=15,
    step=1,
    value=6,
    description=' ',
    readout=False,
    continuous_update=True,
    style=slider_style,
    layout=slider_layout
)

# ============================================================
# CURRENT VALUE LABELS
# ============================================================

dt_value = HTML(
    '<div style="font-family:Arial; font-size:14px; font-weight:bold; color:#0b3d91;">0.020</div>'
)

T_value = HTML(
    '<div style="font-family:Arial; font-size:14px; font-weight:bold; color:#0b3d91;">6.0</div>'
)

M_value = HTML(
    '<div style="font-family:Arial; font-size:14px; font-weight:bold; color:#0b3d91;">100</div>'
)

show_value = HTML(
    '<div style="font-family:Arial; font-size:14px; font-weight:bold; color:#0b3d91;">6</div>'
)

# ============================================================
# UPDATE VALUES
# ============================================================

def update_dt_value(change):
    dt_value.value = f'<div style="font-family:Arial; font-size:14px; font-weight:bold; color:#0b3d91;">{dt_slider.value:.3f}</div>'

def update_T_value(change):
    T_value.value = f'<div style="font-family:Arial; font-size:14px; font-weight:bold; color:#0b3d91;">{T_slider.value:.1f}</div>'

def update_M_value(change):
    M_value.value = f'<div style="font-family:Arial; font-size:14px; font-weight:bold; color:#0b3d91;">{M_slider.value}</div>'

def update_show_value(change):
    show_value.value = f'<div style="font-family:Arial; font-size:14px; font-weight:bold; color:#0b3d91;">{show_slider.value}</div>'

dt_slider.observe(update_dt_value, names='value')
T_slider.observe(update_T_value, names='value')
M_slider.observe(update_M_value, names='value')
show_slider.observe(update_show_value, names='value')

# ============================================================
# CONTROL LABELS
# ============================================================

dt_label = HTML(
    '<div style="font-family:Arial; font-size:14px; font-weight:bold; white-space:nowrap;">Time step Δt:</div>'
)

T_label = HTML(
    '<div style="font-family:Arial; font-size:14px; font-weight:bold; white-space:nowrap;">Final time T:</div>'
)

M_label = HTML(
    '<div style="font-family:Arial; font-size:14px; font-weight:bold; white-space:nowrap;">Realizations M:</div>'
)

show_label = HTML(
    '<div style="font-family:Arial; font-size:14px; font-weight:bold; white-space:nowrap;">Paths shown:</div>'
)

# ============================================================
# CONTROLS GRID
# ============================================================

controls_grid = GridBox(
    children=[
        dt_label, dt_slider, dt_value,
        T_label, T_slider, T_value,
        M_label, M_slider, M_value,
        show_label, show_slider, show_value
    ],
    layout=Layout(
        width='790px',
        grid_template_columns='115px 160px 55px 115px 160px 55px',
        grid_template_rows='34px 34px',
        grid_gap='5px 8px',
        align_items='center',
        overflow='hidden'
    )
)

# ============================================================
# CONTROLS CARD
# ============================================================

controls_card = VBox(
    [
        HTML("""
        <div style="
            font-family:Arial;
            font-size:17px;
            font-weight:bold;
            color:#12388c;
            margin-bottom:5px;
        ">
        Parameters
        </div>
        """),
        controls_grid
    ],
    layout=Layout(
        width='820px',
        padding='10px 14px',
        border='1px solid #d2d2d2',
        overflow='hidden',
        margin='10px 0px 10px 0px'
    )
)

# ============================================================
# RESULT BOX
# ============================================================

result_html = HTML()

# ============================================================
# MAIN FUNCTION
# ============================================================

def plot_wiener_process(dt=0.02, T=6.0, M=100, paths_shown=6):

    # --------------------------------------------------------
    # NUMBER OF STEPS
    # --------------------------------------------------------

    N = int(T / dt)

    N = min(
        N,
        N_max
    )

    t = np.arange(
        1,
        N + 1
    ) * dt

    # --------------------------------------------------------
    # GAUSSIAN INCREMENTS
    #
    # ΔW ~ N(0, Δt)
    # --------------------------------------------------------

    dW = np.sqrt(dt) * Z[:M, :N]

    # --------------------------------------------------------
    # WIENER PROCESS
    # --------------------------------------------------------

    W = np.cumsum(
        dW,
        axis=1
    )

    # --------------------------------------------------------
    # ENSEMBLE STATISTICS
    # --------------------------------------------------------

    estimated_mean = np.mean(
        W,
        axis=0
    )

    estimated_variance = np.var(
        W,
        axis=0
    )

    theoretical_mean = np.zeros_like(
        t
    )

    theoretical_variance = t

    # --------------------------------------------------------
    # FINITE-DIFFERENCE "DERIVATIVE"
    # --------------------------------------------------------

    white_approx = dW[0, :] / dt

    # ========================================================
    # FIXED / CONTROLLED AXIS LIMITS
    # ========================================================

    path_limit = 4.0 * np.sqrt(T)

    mean_limit = 1.5

    variance_limit = 1.20 * T

    noise_limit = 4.0 / np.sqrt(dt)

    # ========================================================
    # FIGURE
    # ========================================================

    fig = plt.figure(
        figsize=(10.4, 7.2)
    )

    gs = fig.add_gridspec(
        2,
        2,
        hspace=0.44,
        wspace=0.28
    )

    ax1 = fig.add_subplot(
        gs[0, 0]
    )

    ax2 = fig.add_subplot(
        gs[0, 1]
    )

    ax3 = fig.add_subplot(
        gs[1, 0]
    )

    ax4 = fig.add_subplot(
        gs[1, 1]
    )

    # ========================================================
    # GRAPH 1:
    # WIENER REALIZATIONS
    # ========================================================

    number_to_show = min(
        paths_shown,
        M
    )

    for m in range(number_to_show):

        ax1.plot(
            t,
            W[m, :],
            linewidth=1.0,
            alpha=0.75
        )

    ax1.axhline(
        0,
        linewidth=0.8
    )

    ax1.set_xlim(
        0,
        T
    )

    ax1.set_ylim(
        -path_limit,
        path_limit
    )

    ax1.set_xlabel(
        'Time t',
        fontsize=11
    )

    ax1.set_ylabel(
        'W(t)',
        fontsize=11
    )

    ax1.set_title(
        'Wiener Process Realizations',
        fontsize=13,
        pad=9
    )

    ax1.tick_params(
        axis='both',
        labelsize=9
    )

    ax1.grid(
        True,
        linestyle=':',
        alpha=0.5
    )

    # ========================================================
    # GRAPH 2:
    # FINITE-DIFFERENCE WHITE-NOISE APPROXIMATION
    # ========================================================

    ax2.plot(
        t,
        white_approx,
        linewidth=0.9
    )

    ax2.axhline(
        0,
        linewidth=0.8
    )

    ax2.set_xlim(
        0,
        T
    )

    ax2.set_ylim(
        -noise_limit,
        noise_limit
    )

    ax2.set_xlabel(
        'Time t',
        fontsize=11
    )

    ax2.set_ylabel(
        'ΔW / Δt',
        fontsize=11
    )

    ax2.set_title(
        'Finite-Difference Approximation to White Noise',
        fontsize=13,
        pad=9
    )

    ax2.tick_params(
        axis='both',
        labelsize=9
    )

    ax2.grid(
        True,
        linestyle=':',
        alpha=0.5
    )

    # ========================================================
    # GRAPH 3:
    # ENSEMBLE MEAN
    # ========================================================

    ax3.plot(
        t,
        estimated_mean,
        linewidth=1.7,
        label='Estimated ensemble mean'
    )

    ax3.plot(
        t,
        theoretical_mean,
        linestyle='--',
        linewidth=2.0,
        label='Theoretical mean = 0'
    )

    ax3.set_xlim(
        0,
        T
    )

    ax3.set_ylim(
        -mean_limit,
        mean_limit
    )

    ax3.set_xlabel(
        'Time t',
        fontsize=11
    )

    ax3.set_ylabel(
        'Mean',
        fontsize=11
    )

    ax3.set_title(
        'Ensemble Mean versus Time',
        fontsize=13,
        pad=9
    )

    ax3.tick_params(
        axis='both',
        labelsize=9
    )

    ax3.grid(
        True,
        linestyle=':',
        alpha=0.5
    )

    ax3.legend(
        loc='upper center',
        bbox_to_anchor=(0.5, -0.20),
        ncol=1,
        fontsize=8
    )

    # ========================================================
    # GRAPH 4:
    # ENSEMBLE VARIANCE
    # ========================================================

    ax4.plot(
        t,
        estimated_variance,
        linewidth=1.7,
        label='Estimated ensemble variance'
    )

    ax4.plot(
        t,
        theoretical_variance,
        linestyle='--',
        linewidth=2.0,
        label='Theoretical variance = t'
    )

    ax4.set_xlim(
        0,
        T
    )

    ax4.set_ylim(
        0,
        variance_limit
    )

    ax4.set_xlabel(
        'Time t',
        fontsize=11
    )

    ax4.set_ylabel(
        'Variance',
        fontsize=11
    )

    ax4.set_title(
        'Ensemble Variance versus Time',
        fontsize=13,
        pad=9
    )

    ax4.tick_params(
        axis='both',
        labelsize=9
    )

    ax4.grid(
        True,
        linestyle=':',
        alpha=0.5
    )

    ax4.legend(
        loc='upper center',
        bbox_to_anchor=(0.5, -0.20),
        ncol=1,
        fontsize=8
    )

    # ========================================================
    # FIGURE SPACING
    # ========================================================

    fig.subplots_adjust(
        left=0.08,
        right=0.97,
        top=0.93,
        bottom=0.15
    )

    plt.show()

    plt.close(fig)

    # ========================================================
    # NUMERICAL RESULTS
    # ========================================================

    result_html.value = f"""
    <div style="
        font-family:Arial, sans-serif;
        font-size:15px;
        line-height:1.42;
        width:930px;
        padding:10px 14px;
        border:1px solid #d7c38d;
        background:#fffbed;
        box-sizing:border-box;
    ">

    <b>Increment variance:</b>
    Var{{ΔW}} = Δt = {dt:.4f}

    &nbsp;&nbsp;&nbsp;

    <b>Samples:</b>
    N = {N}

    <br>

    <b>At t = {t[-1]:.2f}:</b>
    theoretical E{{W(t)}} = 0

    &nbsp;&nbsp;&nbsp;

    theoretical Var{{W(t)}} = {t[-1]:.4f}

    </div>
    """

# ============================================================
# INTERACTIVE OUTPUT
# ============================================================

output = interactive_output(
    plot_wiener_process,
    {
        'dt': dt_slider,
        'T': T_slider,
        'M': M_slider,
        'paths_shown': show_slider
    }
)

# ============================================================
# INTERPRETATION
# ============================================================

interpretation = HTML("""
<div style="
    font-family:Arial, sans-serif;
    font-size:15px;
    line-height:1.42;
    width:1050px;
    padding:11px 15px;
    border:1px solid #c8dfce;
    background:#f8fcf9;
    box-sizing:border-box;
    margin-top:6px;
">

<div style="
    font-size:18px;
    font-weight:bold;
    color:#197b35;
    margin-bottom:6px;
">
Interpretation of the Results
</div>

<div style="margin-bottom:4px;">
The Wiener process is obtained by accumulating independent Gaussian increments with variance Δt.
</div>

<div style="margin-bottom:4px;">
Its ensemble mean remains zero, while its variance grows linearly with time: Var{W(t)} = t.
</div>

<div style="margin-bottom:4px;">
Therefore the Wiener process is not WSS, since its second-order statistics depend explicitly on time.
</div>

<div>
The quantity ΔW/Δt becomes increasingly irregular as Δt decreases, illustrating the formal connection dW/dt = white noise.
</div>

</div>
""")

# ============================================================
# COMPLETE NOTEBOOK
# ============================================================

main_layout = VBox(
    [
        documentation,
        controls_card,
        output,
        result_html,
        interpretation
    ],
    layout=Layout(
        width='1050px',
        overflow='hidden'
    )
)

display(main_layout)